# Dual Camera Calibration — IMX219 (Cam0) + OV9281 (Cam1)

**Checkerboard**: 9×13 squares, 30 mm edge → **8×12 inner corners**  
**Pipeline**: Mono fisheye calibration → Stereo calibration → Stereo rectification → 3-D triangulation check  
**Sampling**: every Nth frame to collect ≈80 valid detections per camera

In [1]:
import cv2
import numpy as np
import msgpack as mp
import msgpack_numpy as mpn
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import toml
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "toml"], check=True)
    import toml

print("OpenCV:", cv2.__version__)

OpenCV: 4.12.0


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# CONFIG  – edit paths here if needed
# ──────────────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CALIB_DIR  = os.path.join(BASE_DIR, "data", "dual_data",
                           "dual_cam_calibration_checker_sz_30mm")
OUTPUT_DIR = os.path.join(os.getcwd(), "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAM0_FILE = os.path.join(CALIB_DIR, "cam0_color.msgpack")   # IMX219  BGR  1640x1232
CAM1_FILE = os.path.join(CALIB_DIR, "cam1_ov9281.msgpack")  # OV9281  gray 1280x800

CAM0_SIZE = (1640, 1232)   # (width, height)
CAM1_SIZE = (1280, 800)

# Checkerboard  9x13 squares  →  8x12 inner corners
PATTERN_COLS = 8
PATTERN_ROWS = 12
SQUARE_MM    = 30.0           # mm

TARGET_SAMPLES = 80           # frames to collect per camera

# OpenCV parameters
CRITERIA   = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
CB_FLAGS   = (cv2.CALIB_CB_ADAPTIVE_THRESH |
              cv2.CALIB_CB_FAST_CHECK      |
              cv2.CALIB_CB_NORMALIZE_IMAGE)
FISH_FLAGS = (cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC |
              cv2.fisheye.CALIB_CHECK_COND           |
              cv2.fisheye.CALIB_FIX_SKEW)

# 3-D object points for one board pose  →  shape (N, 1, 3)
N_CORNERS = PATTERN_COLS * PATTERN_ROWS
OBJP = np.zeros((N_CORNERS, 1, 3), np.float64)
OBJP[:, 0, :2] = (
    np.mgrid[0:PATTERN_COLS, 0:PATTERN_ROWS]
    .T.reshape(-1, 2) * SQUARE_MM
)

for label, path in [("CAM0", CAM0_FILE), ("CAM1", CAM1_FILE)]:
    status = "OK" if os.path.exists(path) else "NOT FOUND!"
    print(f"{label}: {status}  →  {path}")
print(f"Board: {PATTERN_COLS}x{PATTERN_ROWS} corners, {SQUARE_MM} mm | N={N_CORNERS}")

CAM0: NOT FOUND!  →  e:\CMC\pyprojects\programs_rpi\NOARK_backbone\data\dual_data\dual_cam_calibration_checker_sz_30mm\cam0_color.msgpack
CAM1: OK  →  e:\CMC\pyprojects\programs_rpi\NOARK_backbone\data\dual_data\dual_cam_calibration_checker_sz_30mm\cam1_ov9281.msgpack
Board: 8x12 corners, 30.0 mm | N=96


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────

def count_frames(path):
    n = 0
    with open(path, "rb") as f:
        for _ in mp.Unpacker(f, object_hook=mpn.decode):
            n += 1
    return n


def detect_corners(gray):
    """findChessboardCorners + SubPix.  Returns (found, corners (N,1,2))."""
    ret, c = cv2.findChessboardCorners(
        gray, (PATTERN_COLS, PATTERN_ROWS), CB_FLAGS)
    if ret:
        c = cv2.cornerSubPix(gray, c, (11, 11), (-1, -1), CRITERIA)
    return ret, c


def collect_corners(path, is_color, size, step, label):
    """Stream `path`, sample every `step` frames, return (objpts, imgpts, vis_list)."""
    objpts, imgpts, vis = [], [], []
    with open(path, "rb") as f:
        pbar = tqdm(total=TARGET_SAMPLES, desc=f"{label} corner detect")
        for i, frame in enumerate(mp.Unpacker(f, object_hook=mpn.decode)):
            if i % step != 0:
                continue
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if is_color else frame
            ret, corners = detect_corners(gray)
            if ret:
                objpts.append(OBJP.copy())
                imgpts.append(corners)
                if len(vis) < 4:
                    vis.append((frame.copy(), gray.copy(), corners.copy()))
                pbar.update(1)
            if len(objpts) >= TARGET_SAMPLES:
                break
        pbar.close()
    print(f"{label}: {len(objpts)} valid frames")
    return objpts, imgpts, vis


def fisheye_calibrate(objpts, imgpts, size, label):
    """Run cv2.fisheye.calibrate. Returns (rms, K, D)."""
    K = np.zeros((3, 3))
    D = np.zeros((4, 1))
    rms, K, D, rvecs, tvecs = cv2.fisheye.calibrate(
        objpts, imgpts, size, K, D, flags=FISH_FLAGS)
    print(f"{label}  RMS: {rms:.4f} px")
    print(f"  K =\n{K}")
    print(f"  D = {D.ravel()}")
    return rms, K, D


def show_undistort(frame, gray, K, D, size, title):
    """Side-by-side: original  |  fisheye-undistorted."""
    m1, m2 = cv2.fisheye.initUndistortRectifyMap(
        K, D, np.eye(3), K, size, cv2.CV_16SC2)
    undist = cv2.remap(frame if frame.ndim == 3 else gray, m1, m2,
                       cv2.INTER_LINEAR, cv2.BORDER_CONSTANT)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, img, t in zip(axes,
                           [frame if frame.ndim == 3 else gray, undist],
                           ["Original", "Undistorted"]):
        disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
        ax.imshow(disp, cmap="gray" if img.ndim == 2 else None)
        ax.set_title(t); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()


def ndarray_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: ndarray_to_list(v) for k, v in obj.items()}
    return obj


def save_toml(path, data):
    with open(path, "w") as f:
        toml.dump(ndarray_to_list(data), f)
    print(f"Saved → {path}")

---
## Section 1 — IMX219 (Cam0) Mono Fisheye Calibration
Color BGR, 1640×1232

In [ ]:
print("Counting cam0 frames...")
n0 = count_frames(CAM0_FILE)
step0 = max(1, n0 // TARGET_SAMPLES)
print(f"Total: {n0}  |  step = {step0}")

objpts0, imgpts0, vis0 = collect_corners(
    CAM0_FILE, is_color=True, size=CAM0_SIZE, step=step0, label="Cam0")

In [ ]:
rms0, K0, D0 = fisheye_calibrate(objpts0, imgpts0, CAM0_SIZE, "Cam0 (IMX219)")

In [ ]:
if vis0:
    frame, gray, corners = vis0[0]
    annotated = cv2.drawChessboardCorners(
        gray.copy(), (PATTERN_COLS, PATTERN_ROWS), corners, True)
    plt.figure(figsize=(7, 5))
    plt.imshow(annotated, cmap="gray"); plt.title("Cam0 detected corners"); plt.axis("off")
    plt.show()
    show_undistort(frame, gray, K0, D0, CAM0_SIZE, "IMX219 undistortion check")

In [ ]:
save_toml(os.path.join(OUTPUT_DIR, "cam0_calib.toml"), {
    "calibration": {
        "camera_matrix":       K0,
        "dist_coeffs":         D0,
        "reprojection_error":  float(rms0),
        "method":              "fisheye",
        "pattern_size":        [PATTERN_COLS, PATTERN_ROWS],
        "square_size_mm":      SQUARE_MM,
        "n_frames_used":       len(objpts0),
        "notes":               "IMX219 160° FOV, 1640x1232",
    },
    "camera": {"model": "IMX219", "resolution": list(CAM0_SIZE)},
})

---
## Section 2 — OV9281 (Cam1) Mono Fisheye Calibration
Grayscale Y-plane, 1280×800

In [ ]:
print("Counting cam1 frames...")
n1 = count_frames(CAM1_FILE)
step1 = max(1, n1 // TARGET_SAMPLES)
print(f"Total: {n1}  |  step = {step1}")

objpts1, imgpts1, vis1 = collect_corners(
    CAM1_FILE, is_color=False, size=CAM1_SIZE, step=step1, label="Cam1")

In [ ]:
rms1, K1, D1 = fisheye_calibrate(objpts1, imgpts1, CAM1_SIZE, "Cam1 (OV9281)")

In [ ]:
if vis1:
    frame1, gray1, corners1 = vis1[0]
    annotated1 = cv2.drawChessboardCorners(
        gray1.copy(), (PATTERN_COLS, PATTERN_ROWS), corners1, True)
    plt.figure(figsize=(7, 5))
    plt.imshow(annotated1, cmap="gray")
    plt.title("Cam1 detected corners"); plt.axis("off"); plt.show()
    show_undistort(gray1, gray1, K1, D1, CAM1_SIZE, "OV9281 undistortion check")

In [ ]:
save_toml(os.path.join(OUTPUT_DIR, "cam1_calib.toml"), {
    "calibration": {
        "camera_matrix":      K1,
        "dist_coeffs":        D1,
        "reprojection_error": float(rms1),
        "method":             "fisheye",
        "pattern_size":       [PATTERN_COLS, PATTERN_ROWS],
        "square_size_mm":     SQUARE_MM,
        "n_frames_used":      len(objpts1),
        "notes":              "OV9281 160° FOV, 1280x800, grayscale",
    },
    "camera": {"model": "OV9281", "resolution": list(CAM1_SIZE)},
})

---
## Section 3 — Stereo Calibration
Find synchronized frame pairs where **both** cameras detect the board, then run `cv2.fisheye.stereoCalibrate` with intrinsics fixed.

In [ ]:
# Use the step from the larger camera for a consistent stride
STEREO_STEP = max(step0, step1)

s_objpts, s_imgpts0, s_imgpts1 = [], [], []
stereo_vis = []   # (bgr0, gray1, c0, c1)

with open(CAM0_FILE, "rb") as f0, open(CAM1_FILE, "rb") as f1:
    unp0 = mp.Unpacker(f0, object_hook=mpn.decode)
    unp1 = mp.Unpacker(f1, object_hook=mpn.decode)
    pbar = tqdm(desc="Stereo pairs", unit="pair")
    for i, (fr0, fr1) in enumerate(zip(unp0, unp1)):
        if i % STEREO_STEP != 0:
            continue
        gr0 = cv2.cvtColor(fr0, cv2.COLOR_BGR2GRAY)
        gr1 = fr1  # already gray
        r0, c0 = detect_corners(gr0)
        r1, c1 = detect_corners(gr1)
        if r0 and r1:
            s_objpts.append(OBJP.copy())
            s_imgpts0.append(c0)
            s_imgpts1.append(c1)
            if len(stereo_vis) < 3:
                stereo_vis.append((fr0.copy(), gr1.copy(), c0.copy(), c1.copy()))
            pbar.update(1)
        if len(s_objpts) >= TARGET_SAMPLES:
            break
    pbar.close()

print(f"Synchronized pairs: {len(s_objpts)}")

In [ ]:
# Stereo calibrate (intrinsics fixed)
# imageSize: use cam0 reference; R and T are pure outputs
R_stereo = np.zeros((3, 3))
T_stereo = np.zeros((3, 1))

rms_stereo, K0_s, D0_s, K1_s, D1_s, R_stereo, T_stereo = cv2.fisheye.stereoCalibrate(
    s_objpts,
    s_imgpts0,
    s_imgpts1,
    K0.copy(), D0.copy(),
    K1.copy(), D1.copy(),
    CAM0_SIZE,
    R_stereo, T_stereo,
    flags=cv2.fisheye.CALIB_FIX_INTRINSIC
)

print(f"Stereo RMS: {rms_stereo:.4f} px")
print(f"R =\n{R_stereo}")
print(f"T = {T_stereo.ravel()}")   # in mm (same units as square_mm)

In [ ]:
save_toml(os.path.join(OUTPUT_DIR, "stereo_calib.toml"), {
    "stereo_calibration": {
        "R":                    R_stereo,
        "T":                    T_stereo,
        "reprojection_error":   float(rms_stereo),
        "n_pairs_used":         len(s_objpts),
        "reference_image_size": list(CAM0_SIZE),
        "notes":                "T in mm; cam0=IMX219 ref frame",
    },
    "camera0": {"model": "IMX219", "resolution": list(CAM0_SIZE)},
    "camera1": {"model": "OV9281", "resolution": list(CAM1_SIZE)},
})

---
## Section 4 — Stereo Rectification
Compute `R1, R2, P1, P2, Q` and build per-camera undistort+rectify maps.  
Verify with an epipolar-line overlay: horizontal rows in both images should align.

In [ ]:
# Use cam0 size as the common output resolution.
# The OV9281 frame can be scaled up / or use CAM1_SIZE as newImageSize.
RECT_SIZE = CAM0_SIZE   # (width, height)

R1 = np.zeros((3, 3))
R2 = np.zeros((3, 3))
P1 = np.zeros((3, 4))
P2 = np.zeros((3, 4))
Q  = np.zeros((4, 4))

(
    R1, R2, P1, P2, Q
) = cv2.fisheye.stereoRectify(
    K0, D0, K1, D1,
    CAM0_SIZE,          # imageSize
    R_stereo, T_stereo,
    R1, R2, P1, P2, Q,
    flags=cv2.CALIB_ZERO_DISPARITY,
    newImageSize=RECT_SIZE
)

# Rectification maps for cam0  (at CAM0_SIZE)
map0_x, map0_y = cv2.fisheye.initUndistortRectifyMap(
    K0, D0, R1, P1, CAM0_SIZE, cv2.CV_16SC2)

# Rectification maps for cam1  (at CAM1_SIZE, using R2 from stereoRectify)
# We scale P2 to cam1 native resolution.
sx = CAM1_SIZE[0] / CAM0_SIZE[0]
sy = CAM1_SIZE[1] / CAM0_SIZE[1]
P2_scaled = P2.copy()
P2_scaled[0] *= sx
P2_scaled[1] *= sy

map1_x, map1_y = cv2.fisheye.initUndistortRectifyMap(
    K1, D1, R2, P2_scaled, CAM1_SIZE, cv2.CV_16SC2)

print("P1:\n", P1)
print("P2:\n", P2)
print("Q:\n",  Q)

In [ ]:
# ── Visualize: epipolar lines ─────────────────────────────────────────────
if stereo_vis:
    fr0, gr1, c0, c1 = stereo_vis[0]
    rect0 = cv2.remap(fr0, map0_x, map0_y, cv2.INTER_LINEAR)
    # resize gr1 canvas to CAM0_SIZE for display only
    rect1_raw = cv2.remap(gr1, map1_x, map1_y, cv2.INTER_LINEAR)
    rect1_disp = cv2.resize(rect1_raw, CAM0_SIZE)
    rect1_bgr  = cv2.cvtColor(rect1_disp, cv2.COLOR_GRAY2BGR)

    # draw horizontal epipolar lines every 80 px
    for y in range(0, CAM0_SIZE[1], 80):
        cv2.line(rect0, (0, y), (CAM0_SIZE[0], y), (0, 255, 0), 1)
        cv2.line(rect1_bgr, (0, y), (CAM0_SIZE[0], y), (0, 255, 0), 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].imshow(cv2.cvtColor(rect0, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Rectified Cam0 (IMX219)"); axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(rect1_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Rectified Cam1 (OV9281, scaled)"); axes[1].axis("off")
    fig.suptitle("Epipolar lines — should be horizontal in both images")
    plt.tight_layout(); plt.show()
else:
    print("No stereo vis frames — run Section 3 first.")

In [ ]:
save_toml(os.path.join(OUTPUT_DIR, "stereo_rectify.toml"), {
    "rectification": {
        "R1":              R1,
        "R2":              R2,
        "P1":              P1,
        "P2":              P2,
        "Q":               Q,
        "rect_size":       list(RECT_SIZE),
        "notes":           "Use P1/P2 for triangulatePoints after undistortPoints",
    }
})

---
## Section 5 — 3-D Triangulation Verification
Undistort detected corners from a synchronized pair, triangulate with `cv2.triangulatePoints`,  
then compare recovered inter-corner distances to the known **30 mm** grid spacing.

In [ ]:
if not s_imgpts0 or not s_imgpts1:
    raise RuntimeError("Run Section 3 first to collect stereo pairs.")

# Pick the first good stereo pair
pair_idx = 0
pts0_raw = s_imgpts0[pair_idx]  # (N, 1, 2)
pts1_raw = s_imgpts1[pair_idx]

# Undistort using fisheye model  →  normalised (undistorted) image coords
pts0_und = cv2.fisheye.undistortPoints(pts0_raw, K0, D0)  # (N, 1, 2)
pts1_und = cv2.fisheye.undistortPoints(pts1_raw, K1, D1)

# Reshape to (2, N) for triangulatePoints
pts0_2d = pts0_und.reshape(-1, 2).T
pts1_2d = pts1_und.reshape(-1, 2).T

# Projection matrices in normalised space (after undistortPoints K→identity)
# Cam0 is the reference frame: P0 = [I | 0]
P0_norm = np.eye(3, 4, dtype=np.float64)
# Cam1: P1 = K_norm * [R | T]  but after undistortPoints normalisation K=I
P1_norm = np.hstack([R_stereo, T_stereo])   # (3, 4)

# Triangulate  →  homogeneous 4xN
pts_4d = cv2.triangulatePoints(P0_norm, P1_norm, pts0_2d, pts1_2d)
pts_3d = (pts_4d[:3] / pts_4d[3]).T   # (N, 3)  in mm

print(f"Triangulated {pts_3d.shape[0]} corners")
print(f"Depth range (Z): {pts_3d[:, 2].min():.1f} – {pts_3d[:, 2].max():.1f} mm")

In [ ]:
# ── Verify distances against 30 mm ground truth ───────────────────────────
# Horizontal neighbours  →  expected = SQUARE_MM
h_errors, v_errors = [], []

for r in range(PATTERN_ROWS):
    for c in range(PATTERN_COLS - 1):
        i0 = r * PATTERN_COLS + c
        i1 = i0 + 1
        d = np.linalg.norm(pts_3d[i0] - pts_3d[i1])
        h_errors.append(abs(d - SQUARE_MM))

for r in range(PATTERN_ROWS - 1):
    for c in range(PATTERN_COLS):
        i0 = r * PATTERN_COLS + c
        i1 = i0 + PATTERN_COLS
        d = np.linalg.norm(pts_3d[i0] - pts_3d[i1])
        v_errors.append(abs(d - SQUARE_MM))

h_err = np.array(h_errors)
v_err = np.array(v_errors)

print(f"Horizontal distance error  mean={h_err.mean():.2f} mm  std={h_err.std():.2f} mm")
print(f"Vertical   distance error  mean={v_err.mean():.2f} mm  std={v_err.std():.2f} mm")

# 3-D scatter
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(pts_3d[:, 0], pts_3d[:, 1], pts_3d[:, 2], c='steelblue', s=20)
ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
ax.set_title('Triangulated checkerboard corners')
plt.tight_layout(); plt.show()

print("\nCalibration complete!")
print(f"  cam0_calib.toml   RMS={rms0:.4f}")
print(f"  cam1_calib.toml   RMS={rms1:.4f}")
print(f"  stereo_calib.toml RMS={rms_stereo:.4f}")
print(f"  stereo_rectify.toml saved")